In [ ]:
!pip install -q qiskit qiskit-aer gpu qiskit-machine-learning torch scikit-learn



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '[gpu]': Expected package name at the start of dependency specifier
    [gpu]
    ^


In [4]:
pip install qiskit-aer gpu

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement gpu (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for gpu


In [2]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report

import torch
import torch.nn as nn
import torch.optim as optim

from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap, TwoLocal
from qiskit.quantum_info import Pauli
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

# For reproducibility & dtype consistency
np.random.seed(42)
torch.manual_seed(42)
torch.set_default_dtype(torch.float32)


Data: Breast Cancer, scale, PCA → 4 features

In [3]:
# Load breast cancer dataset
data = load_breast_cancer()
X = data.data          # shape (569, 30)
y = data.target        # 0 / 1

print("Original shape:", X.shape)
print("Original #features:", X.shape[1])

# Train/test split
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled  = scaler.transform(X_test_full)

# PCA -> 4 components (4 < 30 original features)
NUM_QUBITS = 4
pca = PCA(n_components=NUM_QUBITS, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled).astype(np.float32)
X_test_pca  = pca.transform(X_test_scaled).astype(np.float32)

print("After PCA, train shape:", X_train_pca.shape)
print("Original features (30) > qubits (4): condition satisfied.")


Original shape: (569, 30)
Original #features: 30
After PCA, train shape: (398, 4)
Original features (30) > qubits (4): condition satisfied.


Build the circuit: ZZFeatureMap (QSVM feature map) + small ansatz

In [4]:
# QSVM feature map (data encoding)
feature_map = ZZFeatureMap(
    feature_dimension=NUM_QUBITS,
    reps=1,
    entanglement='linear'
)

# Small variational ansatz on top
ansatz = TwoLocal(
    num_qubits=NUM_QUBITS,
    rotation_blocks='ry',
    entanglement_blocks='cx',
    entanglement='linear',
    reps=1,              # keep shallow
)

# Full VQC circuit: V(θ) * Uφ(x)
vqc_circuit = feature_map.compose(ansatz)

print("VQC circuit:")
print(vqc_circuit.decompose().draw(fold=60))


VQC circuit:
     ┌───┐┌─────────────┐     »
q_0: ┤ H ├┤ P(2.0*x[0]) ├──■──»
     ├───┤├─────────────┤┌─┴─┐»
q_1: ┤ H ├┤ P(2.0*x[1]) ├┤ X ├»
     ├───┤├─────────────┤└───┘»
q_2: ┤ H ├┤ P(2.0*x[2]) ├─────»
     ├───┤├─────────────┤     »
q_3: ┤ H ├┤ P(2.0*x[3]) ├─────»
     └───┘└─────────────┘     »
«                                          ┌──────────┐»
«q_0: ──────────────────────────────────■──┤ Ry(θ[0]) ├»
«     ┌──────────────────────────────┐┌─┴─┐└──────────┘»
«q_1: ┤ P(2.0*(π - x[0])*(π - x[1])) ├┤ X ├─────■──────»
«     └──────────────────────────────┘└───┘   ┌─┴─┐    »
«q_2: ────────────────────────────────────────┤ X ├────»
«                                             └───┘    »
«q_3: ─────────────────────────────────────────────────»
«                                                      »
«                                                      »
«q_0: ─────────────────────────────────────────────────»
«                                          ┌──────────┐»
«q_1: ─────────

Building EstimatorQNN + PyTorch model

In [5]:
# Backend primitive (statevector simulator)
estimator = AerEstimator()
observable = Pauli("Z" + "I" * (NUM_QUBITS - 1))   # measure Z on first qubit

# Split circuit parameters into input (x) and weights (θ)
input_params  = list(feature_map.parameters)       # data parameters
weight_params = list(ansatz.parameters)            # trainable parameters

qnn = EstimatorQNN(
    estimator=estimator,
    circuit=vqc_circuit,
    input_params=input_params,
    weight_params=weight_params,
    observables=observable,
    input_gradients=True,
)

class VQCModel(nn.Module):
    """
    Wrap EstimatorQNN in a PyTorch module:
    QNN -> expectation in [-1,1] -> Linear -> Sigmoid -> P(y=1)
    """
    def __init__(self, qnn):
        super().__init__()
        self.qnn = TorchConnector(qnn)
        self.fc = nn.Linear(1, 1)

    def forward(self, x):
        exp_val = self.qnn(x)           # (batch, 1)
        logits  = self.fc(exp_val)      # (batch, 1)
        return torch.sigmoid(logits)    # (batch, 1)


C:\Users\hisham.fawad\AppData\Local\Temp\ipykernel_7356\2244486659.py:9: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  qnn = EstimatorQNN(


Training VQC

In [6]:
# Convert data to torch tensors
X_train_t = torch.tensor(X_train_pca, dtype=torch.float32)
y_train_t = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
X_test_t  = torch.tensor(X_test_pca,  dtype=torch.float32)

model = VQCModel(qnn)
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

EPOCHS = 5
BATCH_SIZE = 16

for epoch in range(EPOCHS):
    # shuffle
    perm = torch.randperm(len(X_train_t))
    X_train_t = X_train_t[perm]
    y_train_t = y_train_t[perm]

    model.train()
    total_loss = 0.0

    for i in range(0, len(X_train_t), BATCH_SIZE):
        xb = X_train_t[i:i+BATCH_SIZE]
        yb = y_train_t[i:i+BATCH_SIZE]

        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} - loss: {total_loss:.4f}")


Epoch 1/5 - loss: 20.8366
Epoch 2/5 - loss: 19.2655
Epoch 3/5 - loss: 18.1872
Epoch 4/5 - loss: 17.5072
Epoch 5/5 - loss: 17.0722


In [7]:
model.eval()
with torch.no_grad():
    probs = model(X_test_t).numpy().flatten()

y_pred = (probs > 0.5).astype(int)

print("\nVQC Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))



VQC Accuracy: 0.6257309941520468

Classification report:
              precision    recall  f1-score   support

           0     0.5000    0.0312    0.0588        64
           1     0.6287    0.9813    0.7664       107

    accuracy                         0.6257       171
   macro avg     0.5644    0.5063    0.4126       171
weighted avg     0.5806    0.6257    0.5016       171



# """
# VQC Credit Card Fraud Detection - SIMPLIFIED VERSION (TRACKED)
# No external resampling libraries needed
# Ready to run on Kaggle/Colab immediately

# Tracking added: timing, hw info, memory, VQC iteration losses, JSON output
# """

In [1]:


import numpy as np
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from time import time
import json
import platform
import psutil
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)
from sklearn.decomposition import PCA
# Qiskit imports (keep as original)
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import COBYLA
from qiskit_machine_learning.algorithms import VQC
from qiskit.primitives import Sampler


In [2]:

# Print header (unchanged)
print("=" * 80)
print("VQC CREDIT CARD FRAUD DETECTION")
print("=" * 80)


VQC CREDIT CARD FRAUD DETECTION


In [3]:

# ---------------------------
# Tracking structure
# ---------------------------
TRACK = {
    "meta": {
        "script_start": datetime.now().isoformat(),
        "script_name": os.path.basename(__file__) if '__file__' in globals() else "vqc_tracked.py"
    },
    "hardware": {},
    "timings": {},
    "vqc_iterations": [],
    "memory": {},
    "results": {},
}

def record_hw_info():
    try:
        TRACK["hardware"] = {
            "platform": platform.platform(),
            "processor": platform.processor(),
            "cpu_count_logical": psutil.cpu_count(logical=True),
            "cpu_count_physical": psutil.cpu_count(logical=False),
            "total_ram_gb": round(psutil.virtual_memory().total / (1024**3), 2),
            "python_version": platform.python_version(),
            "cwd": os.getcwd()
        }
    except Exception as e:
        TRACK["hardware"]["error"] = str(e)

def snapshot_memory(label):
    try:
        mem = psutil.virtual_memory()
        TRACK["memory"][label] = {
            "timestamp": datetime.now().isoformat(),
            "used_gb": round(mem.used / (1024**3), 3),
            "available_gb": round(mem.available / (1024**3), 3),
            "percent": mem.percent
        }
    except Exception:
        TRACK["memory"][label] = {"error": "psutil failed"}

# call hardware record immediately
record_hw_info()
snapshot_memory("start")


In [4]:

# ============================================
# DATA LOADING & PREPROCESSING
# ============================================

def load_data(filepath='creditcard.csv'):
    """Load and preprocess data"""
    t0 = time()
    print("\n[STEP 1] DATA LOADING")
    print("-" * 80)

    df = pd.read_csv(filepath)
    print(f"Dataset: {df.shape[0]:,} samples × {df.shape[1]} features")
    print(f"Fraud: {df['Class'].sum():,} ({df['Class'].sum()/len(df)*100:.3f}%)")

    X = df.drop('Class', axis=1).values
    y = df['Class'].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    t1 = time()
    TRACK["timings"]["data_loading"] = round(t1 - t0, 3)
    snapshot_memory("after_load")

    print(f"Features standardized (mean={X_scaled.mean():.6f}, std={X_scaled.std():.6f})")
    print(f"[TRACK] data_loading took {TRACK['timings']['data_loading']} sec")

    return X_scaled, y

def simple_balance(X, y, fraud_multiplier=2, max_normal=5000, random_state=42):
    """Simple balancing: keep all fraud, undersample normal"""
    t0 = time()
    print("\n[STEP 2] BALANCING DATASET")
    print("-" * 80)

    np.random.seed(random_state)

    fraud_idx = np.where(y == 1)[0]
    normal_idx = np.where(y == 0)[0]

    print(f"Before: Fraud={len(fraud_idx)}, Normal={len(normal_idx)}")

    # Keep all fraud samples
    n_normal = min(len(fraud_idx) * fraud_multiplier, max_normal, len(normal_idx))
    selected_normal = np.random.choice(normal_idx, n_normal, replace=False)

    # Combine
    selected_idx = np.concatenate([fraud_idx, selected_normal])
    np.random.shuffle(selected_idx)

    X_balanced = X[selected_idx]
    y_balanced = y[selected_idx]

    t1 = time()
    TRACK["timings"]["balancing"] = round(t1 - t0, 3)
    snapshot_memory("after_balance")

    print(f"After:  Fraud={np.sum(y_balanced==1)}, Normal={np.sum(y_balanced==0)}")
    print(f"Total: {len(y_balanced):,} samples (Ratio 1:{n_normal/len(fraud_idx):.1f})")
    print(f"[TRACK] balancing took {TRACK['timings']['balancing']} sec")

    return X_balanced, y_balanced


In [5]:

# ============================================
# QUANTUM CIRCUITS (unchanged, only inserted minimal tracking)
# ============================================

def create_feature_map(num_qubits=15, num_features=30):
    """Multi-layer encoding: 30 features → 15 qubits"""
    params = ParameterVector('x', num_features)
    qc = QuantumCircuit(num_qubits)

    # Layer 1: First 15 features
    for i in range(min(15, num_features)):
        qc.ry(params[i], i)
        qc.rz(params[i], i)

    # Entanglement
    for i in range(num_qubits - 1):
        qc.cx(i, i + 1)

    # Layer 2: Next 14 features (data reuploading)
    if num_features > 15:
        for i in range(14):
            if 15 + i < num_features:
                qc.ry(params[15 + i], i)
                qc.rz(params[15 + i], i)

    # Entanglement
    for i in range(num_qubits - 1):
        qc.cx(i, i + 1)

    # Layer 3: Last feature
    if num_features == 30:
        qc.p(params[29], num_qubits - 1)

    return qc

def create_ansatz(num_qubits=15, ansatz_type='alternating', reps=3):
    """Create variational ansatz"""
    if ansatz_type == 'hardware_efficient':
        return TwoLocal(
            num_qubits, ['ry', 'rz'], 'cx', 'linear', reps=reps, insert_barriers=False
        )
    elif ansatz_type == 'strongly_entangling':
        return TwoLocal(
            num_qubits, ['ry', 'rz'], 'cx', 'full', reps=2, insert_barriers=False
        )
    else:  # alternating (default)
        ansatz = QuantumCircuit(num_qubits)
        params = ParameterVector('θ', num_qubits * reps * 2)

        idx = 0
        for _ in range(reps):
            for i in range(num_qubits):
                ansatz.ry(params[idx], i)
                idx += 1
            for i in range(num_qubits):
                ansatz.cx(i, (i + 1) % num_qubits)
            for i in range(num_qubits):
                ansatz.rz(params[idx], i)
                idx += 1
            for i in range(num_qubits - 1, -1, -1):
                ansatz.cx(i, (i - 1) % num_qubits)

        return ansatz


In [6]:

# ============================================
# TRAINING (tracking added minimally)
# ============================================

def train_vqc(X_train, y_train, X_test, y_test,
              ansatz_type='alternating', num_qubits=15, max_iter=50):
    """Train VQC model"""
    t0 = time()
    print(f"\n[STEP 3] TRAINING VQC - {ansatz_type.upper()}")
    print("-" * 80)

    feature_map = create_feature_map(num_qubits, X_train.shape[1])
    ansatz = create_ansatz(num_qubits, ansatz_type)

    print(f"Feature Map Depth: {feature_map.depth()}")
    print(f"Ansatz Depth: {ansatz.depth()}")
    print(f"Parameters: {ansatz.num_parameters}")

    # Progress callback
    iteration = [0]
    def callback(weights, obj_val):
        iteration[0] += 1
        rec = {"iter": iteration[0], "loss": float(obj_val)}
        TRACK["vqc_iterations"].append(rec)
        # also capture memory snapshot occasionally
        if iteration[0] % 10 == 0:
            snapshot_memory(f"vqc_iter_{iteration[0]}")
        if iteration[0] % 10 == 0 or iteration[0] == 1:
            print(f"  Iter {iteration[0]:3d}: Loss = {obj_val:.6f}")

    print(f"\nOptimizer: COBYLA (maxiter={max_iter})")
    print("Training started...\n")

    # record pre-training memory/time
    snapshot_memory("before_vqc_fit")
    t_fit_start = time()

    vqc = VQC(
        feature_map=feature_map,
        ansatz=ansatz,
        optimizer=COBYLA(maxiter=max_iter),
        sampler=Sampler(),
        callback=callback
    )

    # Fit (this may raise exceptions on unsupported envs; we keep your try/except in main)
    vqc.fit(X_train, y_train)

    t_fit_end = time()
    train_time = t_fit_end - t_fit_start
    TRACK["timings"]["vqc_training"] = round(train_time, 3)
    snapshot_memory("after_vqc_fit")

    print(f"\n✓ Training completed in {train_time:.1f}s ({train_time/60:.1f} min)")

    # Evaluate
    t_eval0 = time()
    y_pred_test = vqc.predict(X_test)
    t_eval1 = time()
    TRACK["timings"]["vqc_predict"] = round(t_eval1 - t_eval0, 3)
    snapshot_memory("after_vqc_predict")

    results = {
        'ansatz': ansatz_type,
        'time': train_time,
        'accuracy': accuracy_score(y_test, y_pred_test),
        'precision': precision_score(y_test, y_pred_test, zero_division=0),
        'recall': recall_score(y_test, y_pred_test, zero_division=0),
        'f1': f1_score(y_test, y_pred_test, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_pred_test) if len(np.unique(y_pred_test))>1 else 0.0,
        'confusion_matrix': confusion_matrix(y_test, y_pred_test).tolist()
    }

    # keep results summary in TRACK
    TRACK["results"]["vqc"] = results

    print_results(results, y_test, y_pred_test)

    t1 = time()
    TRACK["timings"]["vqc_total"] = round(t1 - t0, 3)

    return vqc, results

def print_results(results, y_test, y_pred):
    """Print results"""
    print("\n" + "=" * 80)
    print(f"RESULTS: {results['ansatz'].upper()}")
    print("=" * 80)
    print(f"Time: {results['time']/60:.1f} min")
    print(f"\nMetrics:")
    print(f"  Accuracy:  {results['accuracy']:.4f}")
    print(f"  Precision: {results['precision']:.4f}")
    print(f"  Recall:    {results['recall']:.4f}")
    print(f"  F1-Score:  {results['f1']:.4f}")
    print(f"  ROC-AUC:   {results['roc_auc']:.4f}")

    cm = np.array(results['confusion_matrix'])
    # ensure cm shape correctness
    if cm.size == 4:
        print(f"\nConfusion Matrix:")
        print(f"              Predicted")
        print(f"              Non-Fraud  Fraud")
        print(f"Actual Non-Fraud  {int(cm[0,0]):6d}  {int(cm[0,1]):6d}")
        print(f"       Fraud      {int(cm[1,0]):6d}  {int(cm[1,1]):6d}")
    else:
        print("Confusion matrix not in expected shape")

    print("\nClassification Report:")
    # avoid errors if single-class predictions
    try:
        print(classification_report(y_test, y_pred, target_names=['Non-Fraud', 'Fraud']))
    except Exception:
        print("Classification report failed (single-class prediction?)")


In [7]:

# ============================================
# CLASSICAL BASELINE (with tracking)
# ============================================

def train_baseline(X_train, y_train, X_test, y_test):
    """Train classical baseline"""
    t0 = time()
    print("\n[BASELINE] CLASSICAL ML")
    print("-" * 80)

    from sklearn.ensemble import RandomForestClassifier

    print("Training Random Forest...")
    start = time()
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_time = (time() - start)
    y_pred = rf.predict(X_test)

    TRACK["timings"]["rf_train"] = round(rf_time, 3)
    TRACK["results"]["baseline"] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0)
    }
    snapshot_memory("after_rf")

    print(f"✓ Completed in {rf_time:.1f}s")
    print(f"  Accuracy:  {TRACK['results']['baseline']['accuracy']:.4f}")
    print(f"  Precision: {TRACK['results']['baseline']['precision']:.4f}")
    print(f"  Recall:    {TRACK['results']['baseline']['recall']:.4f}")
    print(f"  F1-Score:  {TRACK['results']['baseline']['f1']:.4f}")

    return rf


In [8]:

# ============================================
# MAIN
# ============================================

def main():
    total_start = time()
    # Configuration (unchanged)
    SAMPLE_SIZE = 50000  # Max normal samples (faster training)
    NUM_QUBITS = 15
    MAX_ITER = 5
    ANSATZ_TYPES = ['hardware_efficient']  # Start with fastest

    print(f"\n⚙️  Configuration:")
    print(f"  Max Normal Samples: {SAMPLE_SIZE:,}")
    print(f"  Qubits: {NUM_QUBITS}")
    print(f"  Max Iterations: {MAX_ITER}")
    print(f"  Ansatz: {', '.join(ANSATZ_TYPES)}")

    # Load data
    t_load = time()
    X, y = load_data()
    TRACK["timings"]["after_load_call"] = round(time() - t_load, 3)

    # Balance
    t_bal = time()
    X_balanced, y_balanced = simple_balance(X, y,
                                           fraud_multiplier=2,
                                           max_normal=SAMPLE_SIZE)
    TRACK["timings"]["after_balance_call"] = round(time() - t_bal, 3)

    # Split
    print("\n[STEP] Splitting data...")
    t_split0 = time()
    X_train, X_test, y_train, y_test = train_test_split(
        X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
    )
    TRACK["timings"]["train_test_split"] = round(time() - t_split0, 3)
    snapshot_memory("after_split")

    print(f"\n📦 Split: Train={len(X_train):,}, Test={len(X_test):,}")

    all_results = []
    for ansatz_type in ANSATZ_TYPES:
        try:
            t_try = time()
            vqc, results = train_vqc(
                X_train, y_train, X_test, y_test,
                ansatz_type=ansatz_type,
                num_qubits=NUM_QUBITS,
                max_iter=MAX_ITER
            )
            TRACK["timings"][f"vqc_{ansatz_type}_total"] = round(time() - t_try, 3)
            all_results.append(results)
        except Exception as e:
            print(f"❌ Error during VQC ({ansatz_type}): {e}")
            TRACK.setdefault("errors", []).append({"stage": "vqc", "ansatz": ansatz_type, "error": str(e)})

    # Train baseline
    try:
        t_rf = time()
        rf = train_baseline(X_train, y_train, X_test, y_test)
        TRACK["timings"]["rf_total"] = round(time() - t_rf, 3)
    except Exception as e:
        print(f"❌ Error during baseline training: {e}")
        TRACK.setdefault("errors", []).append({"stage": "baseline", "error": str(e)})

    # Summary
    if all_results:
        print("\n" + "=" * 80)
        print("SUMMARY")
        print("=" * 80)
        for r in all_results:
            print(f"\n{r['ansatz'].upper()}:")
            print(f"  F1-Score: {r['f1']:.4f}")
            print(f"  Recall:   {r['recall']:.4f} ⭐")
            print(f"  Time:     {r['time']/60:.1f} min")

    total_end = time()
    TRACK["timings"]["total_script"] = round(total_end - total_start, 3)
    TRACK["meta"]["script_end"] = datetime.now().isoformat()

    # Save tracking files (compact + verbose)
    try:
        with open("vqc_tracking.json", "w") as f:
            json.dump(TRACK, f, indent=2)
        # also save a small verbose version containing only iterations if large
        with open("vqc_tracking_verbose.json", "w") as f:
            json.dump({"vqc_iterations": TRACK["vqc_iterations"]}, f)
        print("\n[TRACK] Tracking saved to vqc_tracking.json and vqc_tracking_verbose.json")
    except Exception as e:
        print(f"[TRACK] Failed to save tracking JSON: {e}")

    print("\n" + "=" * 80)
    print("✅ COMPLETED!")
    print("=" * 80)


In [9]:

if __name__ == "__main__":
    main()



⚙️  Configuration:
  Max Normal Samples: 50,000
  Qubits: 15
  Max Iterations: 5
  Ansatz: hardware_efficient

[STEP 1] DATA LOADING
--------------------------------------------------------------------------------
Dataset: 284,807 samples × 30 features
Fraud: 492 (0.173%)
Features standardized (mean=-0.000000, std=1.000000)
[TRACK] data_loading took 2.626 sec

[STEP 2] BALANCING DATASET
--------------------------------------------------------------------------------
Before: Fraud=492, Normal=284315
After:  Fraud=492, Normal=984
Total: 1,476 samples (Ratio 1:2.0)
[TRACK] balancing took 0.013 sec

[STEP] Splitting data...

📦 Split: Train=1,180, Test=296

[STEP 3] TRAINING VQC - HARDWARE_EFFICIENT
--------------------------------------------------------------------------------
Feature Map Depth: 20
Ansatz Depth: 1
Parameters: 120

Optimizer: COBYLA (maxiter=5)
Training started...


✓ Training completed in 4273.4s (71.2 min)

RESULTS: HARDWARE_EFFICIENT
Time: 71.2 min

Metrics:
  Accuracy